# Разметка ЭКГ эксперимента 2

**Статус:** активный производитель кандидатных и принятых вручную R-зубцов.
Автоматическая детекция не создаёт проверенную ЭКГ-разметку.

Ноутбук выделен из исторического `08_Скетч_разметки.ipynb`. Он не определяет
по боковой реограмме механические события или моменты открытия и закрытия
клапанов.


## Входы и научный статус

Входами служат исходный CSV и принятый дыхательный сопроводительный файл той же
записи из [`11.01`](11.01_Разметка_дыхания_эксперимента_2.ipynb). Связь
проверяется по полному SHA-256. Непринятая, отклонённая или созданная старой
версией алгоритма дыхательная разметка блокирует расчёт.

Частота дискретизации оценивается по медиане разностей `TIME_s`. Это расчётная
величина, а не паспортная характеристика прибора. Параметры полосового фильтра,
поиска пиков и уточнения задаются во внешней конфигурации и сохраняются в
результате.

R-зубцы являются результатом алгоритмической детекции до ручного контроля.
Полярность выбирается по записанному ЭКГ-сигналу как диагностическое решение и
также требует проверки.

Расчётное окно электрической систолы строится по рабочей модели

$$
QT=QT_c\sqrt{RR},
$$

где $QT$ — расчётная продолжительность модельного электрического окна;
$RR$ — интервал между соседними принятыми R-зубцами; $QT_c$ — заданный
модельный параметр. Начало Q дополнительно задаётся фиксированным смещением
назад от R-зубца. Такое окно не является измеренной QT-разметкой, механической
систолой или клапанным событием.


In [ ]:
# Импорты и внешняя конфигурация
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp02_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
BREATHING_DIR = DERIVED_ROOT / "exp02" / "annotations" / "breathing"
OUT_DIR = DERIVED_ROOT / "exp02" / "annotations" / "ecg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "TIME_s"
ECG_COL = "ECG_V"
ALGORITHM_VERSION = "exp02-ecg-rpeak-v2"
PARAMETERS = {
    key: float(value)
    for key, value in CONFIG["ecg_detection"].items()
}
REQUIRED_PARAMETERS = {
    "low_hz", "high_hz", "filter_order", "min_rr_s", "peak_height_z",
    "peak_prominence_z", "refine_half_window_s", "qtc_s", "q_lead_s",
    "min_rr_for_qt_s", "default_rr_s",
}
if set(PARAMETERS) != REQUIRED_PARAMETERS:
    raise ValueError("ecg_detection должен содержать полный набор параметров")
for name, value in PARAMETERS.items():
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Параметр {name} должен быть положительным")

subject_items = CONFIG["subjects"]
expected_keys = {
    (item["subject_id"], int(size_mm))
    for item in subject_items
    for size_mm in item["sizes_mm"]
}
expected_count = int(CONFIG["expected_independent_record_count"])
if len(expected_keys) != expected_count:
    raise ValueError("Состав sizes_mm не совпадает с ожидаемым числом записей")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_under_data_root(path):
    resolved = path.expanduser().resolve()
    resolved.relative_to(DATA_ROOT)
    return resolved


def read_record(path):
    frame = pd.read_csv(path, encoding="utf-8")
    missing = {TIME_COL, ECG_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"В CSV отсутствуют обязательные столбцы: {sorted(missing)}")
    for column in (TIME_COL, ECG_COL):
        frame[column] = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(frame[column].to_numpy(dtype=float)).all():
            raise ValueError(f"Столбец {column} содержит нечисловые значения")
    return frame


def sampling_frequency(frame):
    time = frame[TIME_COL].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("TIME_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction


In [ ]:
# Детекция R-зубцов и модельное окно электрической систолы
def detect_rpeaks(ecg, fs_hz, parameters=PARAMETERS):
    low_hz = parameters["low_hz"]
    high_hz = parameters["high_hz"]
    nyquist_hz = fs_hz / 2.0
    if not 0 < low_hz < high_hz < nyquist_hz:
        raise ValueError("Полоса ЭКГ должна находиться ниже частоты Найквиста")

    order = int(parameters["filter_order"])
    if order < 1 or order != parameters["filter_order"]:
        raise ValueError("filter_order должен быть положительным целым")
    b, a = butter(
        order,
        [low_hz / nyquist_hz, high_hz / nyquist_hz],
        btype="band",
    )
    raw = np.asarray(ecg, dtype=float)
    min_samples = 3 * max(len(a), len(b))
    if len(raw) <= min_samples or not np.isfinite(raw).all():
        raise ValueError("ЭКГ слишком короткая или содержит нечисловые значения")
    filtered = filtfilt(b, a, raw)
    scale = float(np.std(filtered))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        raise ValueError("После фильтрации отсутствует вариабельность ЭКГ")
    normalized = filtered / scale

    positive_strength = float(np.percentile(normalized, 99.5))
    negative_strength = float(np.percentile(-normalized, 99.5))
    polarity = 1 if positive_strength >= negative_strength else -1
    signed = polarity * normalized
    candidates, _ = find_peaks(
        signed,
        distance=max(1, int(round(parameters["min_rr_s"] * fs_hz))),
        height=parameters["peak_height_z"],
        prominence=parameters["peak_prominence_z"],
    )

    refined = []
    half_window = max(1, int(round(parameters["refine_half_window_s"] * fs_hz)))
    signed_raw = polarity * raw
    for candidate in candidates:
        start = max(0, candidate - half_window)
        stop = min(len(raw), candidate + half_window + 1)
        refined.append(start + int(np.argmax(signed_raw[start:stop])))
    return np.asarray(sorted(set(refined)), dtype=int), int(polarity)


def model_electrical_systole(rpeaks_s, parameters=PARAMETERS):
    windows = []
    rpeaks_s = np.asarray(rpeaks_s, dtype=float)
    for index, r_time in enumerate(rpeaks_s):
        if index + 1 < len(rpeaks_s):
            rr_s = rpeaks_s[index + 1] - r_time
        elif index > 0:
            rr_s = r_time - rpeaks_s[index - 1]
        else:
            rr_s = parameters["default_rr_s"]
        qt_s = parameters["qtc_s"] * np.sqrt(
            max(rr_s, parameters["min_rr_for_qt_s"])
        )
        q_start_s = r_time - parameters["q_lead_s"]
        t_end_s = q_start_s + qt_s
        windows.append([float(q_start_s), float(t_end_s)])
    return windows


In [ ]:
# Построение отдельных кандидатных ЭКГ-sidecar-файлов
breathing_files = sorted(BREATHING_DIR.glob("*.json"))
breathing_records = []
seen_keys = set()
for breathing_path in breathing_files:
    breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
    if breathing.get("annotation_type") != "breathing":
        continue
    key = (breathing.get("subject_id"), int(breathing.get("size_mm")))
    if key in seen_keys:
        raise ValueError(f"Несколько дыхательных sidecar для {key}")
    seen_keys.add(key)
    if key not in expected_keys:
        raise ValueError(f"Неожиданная дыхательная запись: {key}")
    if breathing.get("algorithm_version") != "exp02-breathing-heuristic-v2":
        raise RuntimeError(f"Устаревшая версия дыхательной разметки: {key}")
    if breathing.get("qc", {}).get("status") != "accepted":
        raise RuntimeError(f"Дыхательная разметка не принята: {key}")
    if not breathing.get("accepted_modes"):
        raise RuntimeError(f"Нет accepted_modes в принятой разметке: {key}")
    breathing_records.append((breathing_path, breathing))

if seen_keys != expected_keys or len(breathing_records) != expected_count:
    missing = sorted(expected_keys - seen_keys)
    raise RuntimeError(
        f"Нужны все {expected_count} принятых дыхательных разметок; "
        f"отсутствуют {missing}"
    )

records = []
record_ids = set()
for breathing_path, breathing in breathing_records:
    source_path = resolve_under_data_root(
        DATA_ROOT / breathing["input"]["relative_path"]
    )
    input_sha256 = sha256_file(source_path)
    if input_sha256 != breathing["input"]["sha256"]:
        raise RuntimeError(
            f"Исходный файл изменился после дыхательной разметки: "
            f"{breathing['record_id']}"
        )
    if breathing["record_id"] != input_sha256[:16]:
        raise RuntimeError("record_id не соответствует SHA-256 исходного CSV")

    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    peak_indices, polarity = detect_rpeaks(
        frame[ECG_COL].to_numpy(dtype=float),
        fs_hz,
    )
    time = frame[TIME_COL].to_numpy(dtype=float)
    candidate_rpeaks_s = [float(time[index]) for index in peak_indices]
    candidate_windows = model_electrical_systole(candidate_rpeaks_s)
    record_id = breathing["record_id"]
    if record_id in record_ids:
        raise ValueError("Повторный record_id среди независимых записей")
    record_ids.add(record_id)

    output = {
        "schema_version": 2,
        "annotation_type": "ecg",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": record_id,
        "subject_id": breathing["subject_id"],
        "size_mm": breathing["size_mm"],
        "input": {
            "relative_path": breathing["input"]["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_TIME_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "upstream_breathing": {
            "sidecar_sha256": sha256_file(breathing_path),
            "algorithm_version": breathing["algorithm_version"],
            "qc_status": breathing["qc"]["status"],
            "accepted_modes": breathing["accepted_modes"],
        },
        "candidate_rpeaks_s": candidate_rpeaks_s,
        "candidate_model_electrical_systole_s": candidate_windows,
        "candidate_polarity": polarity,
        "accepted_polarity": None,
        "rpeaks_s": None,
        "model_electrical_systole_s": None,
        "model_parameters": PARAMETERS,
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
            "history": [],
        },
    }
    output_path = OUT_DIR / f"{record_id}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("input", {}).get("sha256") != input_sha256:
            raise RuntimeError(f"Конфликт входного SHA-256 для {record_id}")
        if existing.get("algorithm_version") != ALGORITHM_VERSION:
            raise RuntimeError(
                f"ЭКГ-sidecar {record_id} создан другой версией алгоритма; "
                "его нужно отдельно пересмотреть или архивировать"
            )
        if (
            existing.get("qc", {}).get("status") == "accepted"
            and (
                not existing.get("rpeaks_s")
                or existing.get("accepted_polarity") not in {-1, 1}
            )
        ):
            raise RuntimeError(f"Принятый ЭКГ-sidecar {record_id} не содержит rpeaks_s или принятую полярность")
        records.append(existing)
        print("Сохранён существующий ЭКГ-sidecar:", record_id, existing["qc"]["status"])
        continue

    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    records.append(output)
    print("Создан кандидат:", record_id, len(candidate_rpeaks_s))

if len(records) != expected_count:
    raise RuntimeError(f"Получено {len(records)} ЭКГ-sidecar вместо {expected_count}")
print("Кандидатных или ранее сохранённых ЭКГ-sidecar-файлов:", len(records))


## Ручной контроль качества и принятие разметки

Для каждой записи необходимо проверить пропущенные и ложные R-зубцы, участки с
артефактами, полярность и пригодность записи. Решение принимается в этом же
ноутбуке через `REVIEW_DECISION`.

Статус `accepted` требует имени проверяющего, явно заданного списка
`accepted_rpeaks_s` и принятой полярности `accepted_polarity`. После
принятия модельные окна пересчитываются по этому списку. Дыхательный статус
проверяется как обязательный вход, но не принимает ЭКГ-разметку
автоматически. Повторный запуск не перезаписывает существующий
сопроводительный файл.


In [ ]:
# Ручной просмотр и явное принятие или отклонение ЭКГ-разметки
CHECK_RECORD_ID = None
WINDOW_S = None
REVIEW_DECISION = None
# Пример решения:
# REVIEW_DECISION = {
#     "record_id": "<record_id>",
#     "status": "accepted",  # accepted или rejected
#     "reviewer": "<reviewer>",
#     "accepted_rpeaks_s": [1.02, 1.84, 2.67],
#     "accepted_polarity": 1,  # 1 или -1
#     "notes": "<основание решения>",
# }


def validate_rpeaks(rpeaks_s, start_s, stop_s):
    values = np.asarray(rpeaks_s, dtype=float)
    if values.ndim != 1 or len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("Нужны не менее двух конечных R-зубцов")
    if np.any(np.diff(values) <= 0):
        raise ValueError("R-зубцы должны быть уникальны и строго упорядочены")
    if values[0] < start_s or values[-1] > stop_s:
        raise ValueError("R-зубцы выходят за границы записи")
    return [float(value) for value in values]


def apply_review(decision):
    record_id = decision["record_id"]
    sidecar_path = OUT_DIR / f"{record_id}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_under_data_root(
        DATA_ROOT / annotation["input"]["relative_path"]
    )
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")

    status = decision["status"]
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError("Нужны статус accepted/rejected и имя проверяющего")

    accepted_rpeaks_s = None
    accepted_polarity = None
    model_windows = None
    if status == "accepted":
        frame = read_record(source_path)
        time = frame[TIME_COL].to_numpy(dtype=float)
        accepted_rpeaks_s = validate_rpeaks(
            decision.get("accepted_rpeaks_s"),
            float(time[0]),
            float(time[-1]),
        )
        accepted_polarity = decision.get("accepted_polarity")
        if accepted_polarity not in {-1, 1}:
            raise ValueError("accepted_polarity должен быть равен 1 или -1")
        model_windows = model_electrical_systole(accepted_rpeaks_s)

    reviewed_at = datetime.now(timezone.utc).isoformat()
    previous_qc = annotation.get("qc", {})
    history = list(previous_qc.get("history", []))
    history.append({
        "status": previous_qc.get("status"),
        "reviewer": previous_qc.get("reviewer"),
        "reviewed_at": previous_qc.get("reviewed_at"),
        "notes": previous_qc.get("notes"),
    })
    annotation["rpeaks_s"] = accepted_rpeaks_s
    annotation["accepted_polarity"] = accepted_polarity
    annotation["model_electrical_systole_s"] = model_windows
    annotation["qc"] = {
        "status": status,
        "reviewer": reviewer,
        "reviewed_at": reviewed_at,
        "notes": decision.get("notes"),
        "history": history,
    }
    sidecar_path.write_text(
        json.dumps(annotation, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return annotation


if REVIEW_DECISION is not None:
    reviewed = apply_review(REVIEW_DECISION)
    print(reviewed["record_id"], reviewed["qc"]["status"])

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID для просмотра конкретной записи.")
else:
    sidecar_path = OUT_DIR / f"{CHECK_RECORD_ID}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_under_data_root(
        DATA_ROOT / annotation["input"]["relative_path"]
    )
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")

    frame = read_record(source_path)
    time = frame[TIME_COL].to_numpy(dtype=float)
    ecg = frame[ECG_COL].to_numpy(dtype=float)
    mask = np.ones(len(time), dtype=bool)
    if WINDOW_S is not None:
        mask = (time >= WINDOW_S[0]) & (time <= WINDOW_S[1])

    rpeaks_s = (
        annotation.get("rpeaks_s")
        if annotation.get("qc", {}).get("status") == "accepted"
        else annotation["candidate_rpeaks_s"]
    )
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time[mask], ecg[mask], color="0.35", linewidth=0.7)
    visible_time = time[mask]
    for r_time in rpeaks_s:
        if len(visible_time) and visible_time[0] <= r_time <= visible_time[-1]:
            axis.axvline(r_time, color="red", linewidth=0.6)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("ЭКГ, В")
    axis.set_title(
        f"{CHECK_RECORD_ID}: ЭКГ-разметка; QC={annotation['qc']['status']}"
    )
    plt.tight_layout()
    plt.show()


## Выход и зависимые этапы

Канонические сопроводительные файлы находятся во внешнем каталоге
`derived/exp02/annotations/ecg/`. Дыхательная и ЭКГ-разметка связываются по
`record_id`, полному SHA-256 исходного CSV и SHA-256 дыхательного
сопроводительного файла, но имеют независимые версии алгоритма и статусы
ручного контроля.

Последующие расчёты должны использовать только поле `rpeaks_s` из ЭКГ-файла
со статусом `accepted`. Поле `candidate_rpeaks_s` предназначено только для
ручного просмотра.
